# Train swinC (SwinUNETR)

5-fold training on preprocessed HFValid (folder with `split.json` attached as input).
Weights are saved to `/kaggle/working`. Needs two 12 h sessions: attach the first output as input and run again to resume.

In [ ]:
!pip install -q monai==1.6.1 einops

In [ ]:
import glob
import json
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk
import torch
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import AttentionUnet, SwinUNETR
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset

T_START = time.time()

## Settings

In [ ]:
NAME = "swinC"
MODEL = dict(arch="SwinUNETR", feature_size=48, use_checkpoint=True)
TRAIN = dict(epochs=80, batch_size=1, lr=1e-4, weight_decay=1e-5, patience=20, seed=42, workers=2, amp=True)
AUGMENT = dict(
    pose_prob=0.8, rotate_deg=(15, 10, 15), shift_vox=4,   # flexion/abduction/rotation in degrees, shift in voxels
    gamma=(0.6, 1.6), contrast=(0.7, 1.3), brightness=0.1,
    blur_prob=0.3, blur_sigma=(0.3, 1.5), noise_prob=0.5, noise_std=0.04,
)
FOLDS = [0, 1, 2, 3, 4]
TIME_BUDGET_H = 11.0

SHAPE = (256, 128, 96)  # (z, y, x)
VOX = 2.0
DRR_SCALE = 6.0

def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) + glob.glob(f"**/{pattern}", recursive=True)
    if not hits:
        raise FileNotFoundError(pattern)
    return Path(hits[0])

SPLIT = find("split.json")
DATA = SPLIT.parent if (SPLIT.parent / "dataset.csv").exists() else find("dataset.csv").parent
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("checkpoints")
OUT.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("data:", DATA, "| split:", SPLIT, "| out:", OUT, "| device:", device)

json.dump(dict(name=NAME, model=MODEL, train=TRAIN, augment=AUGMENT), open(OUT / f"{NAME}_settings.json", "w"), indent=2)

Resume: files from an earlier run of this notebook (attached as an input) are copied back first.

In [ ]:
for f in glob.glob(f"/kaggle/input/**/{NAME}_fold*", recursive=True):
    if not (OUT / Path(f).name).exists():
        shutil.copy(f, OUT)
        print("resumed", Path(f).name)

## Data

In [ ]:
def read(path):
    return sitk.GetArrayFromImage(sitk.ReadImage(str(path)))

folds = json.load(open(SPLIT))
ids = [p for f in folds for p in f]
# DRR rows are flipped so that row index = z (the PNGs have the hip at the top)
DRR = {p: [read(DATA / p / f"{v}.png").astype(np.float32)[::-1] / 65535.0 for v in ("ap", "lat")] for p in ids}
SEG = {p: read(DATA / p / "seg.nii.gz").astype(bool) for p in ids}
_CT = {}

def ct(pid):
    if pid not in _CT:
        _CT[pid] = read(DATA / pid / "ct.nii.gz").astype(np.int16)
    return _CT[pid]

print(len(ids), "patients,", [len(f) for f in folds], "per fold")

In [ ]:
def project(hu):
    """Parallel-beam AP and LAT DRRs of a (z, y, x) HU volume, same formula as the preprocessing."""
    mu = np.clip(0.02 * (1 + hu / 1000.0), 0, None)
    return [np.clip(mu.sum(axis=a) * VOX / DRR_SCALE, 0, 1).astype(np.float32) for a in (1, 2)]


def normalise(im):
    return ((im - im.mean()) / (im.std() + 1e-6)).astype(np.float32)


def fuse(ap, lat):
    """Repeat each DRR along its ray direction and stack: (2, z, y, x)."""
    ap, lat = normalise(ap), normalise(lat)
    return np.stack([np.broadcast_to(ap[:, None, :], SHAPE), np.broadcast_to(lat[:, :, None], SHAPE)]).astype(np.float32)

## Augmentation
Pose: rotate and shift the CT and mask, then re-project the DRRs. Intensity: gamma, contrast, brightness, blur, noise.

In [ ]:
def rotation(rx, ry, rz):
    a, b, c = np.radians([rx, ry, rz])
    Rx = np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])
    Ry = np.array([[np.cos(b), 0, np.sin(b)], [0, 1, 0], [-np.sin(b), 0, np.cos(b)]])
    Rz = np.array([[np.cos(c), -np.sin(c), 0], [np.sin(c), np.cos(c), 0], [0, 0, 1]])
    return Rz @ Ry @ Rx


def random_pose(hu, seg, rng):
    R = rotation(*(rng.uniform(-d, d) for d in AUGMENT["rotate_deg"]))
    P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]])  # (x, y, z) <-> array order (z, y, x)
    M = (P @ R @ P).T  # affine_transform maps output -> input
    c = (np.array(SHAPE) - 1) / 2
    offset = c - M @ (c + rng.uniform(-AUGMENT["shift_vox"], AUGMENT["shift_vox"], 3))
    hu = ndimage.affine_transform(hu.astype(np.float32), M, offset, order=1, cval=-1024.0)
    seg = ndimage.affine_transform(seg.astype(np.float32), M, offset, order=1, cval=0.0) >= 0.5
    return (*project(hu), seg)


def augment_intensity(im, rng):
    a = AUGMENT
    im = np.clip(im, 0, 1) ** rng.uniform(*a["gamma"])
    im = im * rng.uniform(*a["contrast"]) + rng.uniform(-a["brightness"], a["brightness"])
    if rng.random() < a["blur_prob"]:
        im = ndimage.gaussian_filter(im, rng.uniform(*a["blur_sigma"]))
    if rng.random() < a["noise_prob"]:
        im = im + rng.normal(0, rng.uniform(0, a["noise_std"]), im.shape)
    return im.astype(np.float32)


class Cases(Dataset):
    def __init__(self, ids, augment=False):
        self.ids, self.augment = ids, augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]
        ap, lat = DRR[pid]
        seg = SEG[pid]
        if self.augment:
            rng = np.random.default_rng()
            if rng.random() < AUGMENT["pose_prob"]:
                ap, lat, seg = random_pose(ct(pid), seg, rng)
            ap, lat = augment_intensity(ap, rng), augment_intensity(lat, rng)
        return torch.from_numpy(fuse(ap, lat)), torch.from_numpy(seg[None].astype(np.float32))

Check: one augmented sample (AP, LAT, and the mask projected to AP).

In [ ]:
x, y = Cases(ids[:1], augment=True)[0]
fig, ax = plt.subplots(1, 3, figsize=(9, 5))
for a, im, t in zip(ax, [x[0, :, 0], x[1, :, :, 0], y[0].max(1)], ["AP", "LAT", "mask (AP)"]):
    a.imshow(im, cmap="gray", origin="lower"); a.set_title(t); a.axis("off")
plt.show()

## Model and training

In [ ]:
def build_model():
    kw = dict(MODEL)
    arch = {"AttentionUnet": AttentionUnet, "SwinUNETR": SwinUNETR}[kw.pop("arch")]
    return arch(spatial_dims=3, in_channels=2, out_channels=1, **kw).to(device)

print(f"{NAME}: {sum(p.numel() for p in build_model().parameters()) / 1e6:.2f}M parameters")

In [ ]:
def out_of_time():
    return time.time() - T_START > TIME_BUDGET_H * 3600


def train_fold(fold):
    t = TRAIN
    best_p, state_p, done_p = OUT / f"{NAME}_fold{fold}.pt", OUT / f"{NAME}_fold{fold}_state.pt", OUT / f"{NAME}_fold{fold}.json"
    if done_p.exists():
        print(f"fold {fold}: already done")
        return True
    train_ids = [p for k, f in enumerate(folds) if k != fold for p in f]
    train_dl = DataLoader(Cases(train_ids, augment=True), batch_size=t["batch_size"], shuffle=True,
                          num_workers=t["workers"], pin_memory=True, persistent_workers=t["workers"] > 0)
    val_dl = DataLoader(Cases(folds[fold]), batch_size=1, num_workers=t["workers"], pin_memory=True)

    torch.manual_seed(t["seed"] + fold)
    amp = t["amp"] and device == "cuda"
    model = build_model()
    opt = torch.optim.AdamW(model.parameters(), lr=t["lr"], weight_decay=t["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=t["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    loss_fn = DiceCELoss(sigmoid=True)
    dice = DiceMetric(include_background=True, reduction="mean")

    start, best, bad, hist = 1, 0.0, 0, {"loss": [], "dice": []}
    if state_p.exists():
        s = torch.load(state_p, map_location=device, weights_only=False)
        for obj, key in ((model, "model"), (opt, "opt"), (sched, "sched"), (scaler, "scaler")):
            obj.load_state_dict(s[key])
        start, best, bad, hist = s["epoch"] + 1, s["best"], s["bad"], s["hist"]
        print(f"fold {fold}: resuming at epoch {start}")

    for ep in range(start, t["epochs"] + 1):
        if bad >= t["patience"]:
            break
        if out_of_time():
            print(f"fold {fold}: time budget reached at epoch {ep}, run again to resume")
            return False
        t0 = time.time()
        model.train()
        total = 0.0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast(device_type="cuda", enabled=amp):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total += loss.item()
        sched.step()

        model.eval()
        dice.reset()
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                with torch.autocast(device_type="cuda", enabled=amp):
                    dice((torch.sigmoid(model(x)) > 0.5).float(), y)
        vd = dice.aggregate().item()
        hist["loss"].append(total / len(train_dl))
        hist["dice"].append(vd)
        if vd > best:
            best, bad = vd, 0
            torch.save(model.state_dict(), best_p)
        else:
            bad += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), epoch=ep, best=best, bad=bad, hist=hist), state_p)
        print(f"fold {fold} ep {ep:3d}  loss {hist['loss'][-1]:.4f}  val Dice {vd:.4f}  best {best:.4f}  "
              f"({time.time() - t0:.0f}s)", flush=True)

    json.dump({"best_val_dice": best, "epochs_run": len(hist["dice"]), "hist": hist}, open(done_p, "w"))
    state_p.unlink(missing_ok=True)
    print(f"fold {fold}: done, best val Dice {best:.4f}")
    return True

In [ ]:
for fold in FOLDS:
    if not train_fold(fold):
        break

## Results

In [ ]:
done = {f: json.load(open(OUT / f"{NAME}_fold{f}.json")) for f in FOLDS if (OUT / f"{NAME}_fold{f}.json").exists()}
print(f"{len(done)}/{len(FOLDS)} folds finished")
if done:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for f, r in done.items():
        ax[0].plot(r["hist"]["loss"], label=f"fold {f}")
        ax[1].plot(r["hist"]["dice"], label=f"fold {f}")
        print(f"fold {f}: best val Dice {r['best_val_dice']:.4f} after {r['epochs_run']} epochs")
    ax[0].set(title="training loss", xlabel="epoch")
    ax[1].set(title="validation Dice", xlabel="epoch")
    ax[1].legend()
    plt.savefig(OUT / f"{NAME}_curves.png", dpi=120, bbox_inches="tight")
    plt.show()